In [ ]:
%%sql
-- Role-by-role access summary — query count, first/last access per user+role
SELECT
    ROLE_NAME,
    USER_NAME,
    COUNT(*)                 AS query_count,
    MIN(QUERY_START_TIME)    AS first_access,
    MAX(QUERY_START_TIME)    AS last_access
FROM ACCOUNT_ACCESS_HISTORY
GROUP BY ROLE_NAME, USER_NAME
ORDER BY query_count DESC;

## Role-by-Role Access Summary

Who accessed what, how many times, and when — the view CalOptima auditors would use for a HIPAA access review.

In [ ]:
%%sql
-- Last 30 queries across all users and roles
SELECT *
FROM ACCOUNT_ACCESS_HISTORY
WHERE USER_NAME IS NOT NULL
ORDER BY QUERY_START_TIME DESC
LIMIT 30;

## Most Recent Queries — Including the REVOKE Event

The REVOKE and subsequent "not authorized" error from Part 2 appear here after a short ACCOUNT_USAGE propagation lag (~1 min).

In [ ]:
%%sql
-- Build snapshot if not already present (no-op if table exists)
CREATE TABLE IF NOT EXISTS ACCOUNT_ACCESS_HISTORY AS
SELECT
    ah.QUERY_ID,
    ah.QUERY_START_TIME,
    qh.USER_NAME,
    qh.ROLE_NAME,
    LEFT(qh.QUERY_TEXT, 200)                             AS query_preview,
    qh.EXECUTION_STATUS,
    ah.DIRECT_OBJECTS_ACCESSED[0]:objectName::STRING     AS first_object_accessed,
    ah.DIRECT_OBJECTS_ACCESSED[0]:objectDomain::STRING   AS object_domain
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY ah
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh
    ON ah.QUERY_ID = qh.QUERY_ID
WHERE ah.QUERY_START_TIME >= DATEADD('day', -90, CURRENT_TIMESTAMP())
ORDER BY ah.QUERY_START_TIME DESC;

## Pre-built Access History Table

90-day snapshot joining `ACCESS_HISTORY` + `QUERY_HISTORY` from `SNOWFLAKE.ACCOUNT_USAGE`. Created once in setup — queries run instantly, no live ACCOUNT_USAGE latency during the demo.

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
USE DATABASE GOVERNANCE_CA_DEMO;
USE SCHEMA POLICY_STORE;
USE WAREHOUSE WH_XS;

# Security Demo — Part 3: Audit Log

CalOptima RFP 26-038 | Topic 5 — Audit & Compliance

Snowflake captures **every query** — including those that returned 0 rows because of row access policies, and the "not authorized" error from the REVOKE demo.

- CalOptima auditors can query this directly — no log export pipeline needed
- HIPAA requires audit logs of all PHI access: Snowflake provides this natively
- Pre-built as a 90-day snapshot in `GOVERNANCE_CA_DEMO.POLICY_STORE.ACCOUNT_ACCESS_HISTORY`